# 01 — Vector Addition

**Ladder:** Naive (1 element / pid) → blocked + masked → Triton `tl` kernel.

**Study tip.** Master `offsets` + `mask` on a tiny $N$ not divisible by $B$ before any benchmark. This folder is the SPMD foundation for everything else.

Official: [Vector Addition](https://triton-lang.org/main/getting-started/tutorials/01-vector-add.html)

Figures: [`docs/images/`](docs/images/) · visual guide: [`docs/visual-guide.md`](docs/visual-guide.md)

---

# Part I — Mathematics

## 0. The problem, formally

Throughout, write $[N] := \{0,1,\dots,N-1\}$ and let $b$ be the element size in bytes
($b=4$ for `float32`, $b=2$ for `float16`/`bfloat16`).

**Definition 1 (elementwise addition).** For $x,y\in\mathbb{R}^{N}$, the vector
$z = x+y\in\mathbb{R}^{N}$ is defined componentwise by

$$
z_i = x_i + y_i,
\qquad i\in[N].
\tag{1.1}
$$

The map $T(x,y)=x+y$ from
$\mathbb{R}^{N}\times\mathbb{R}^{N}\to\mathbb{R}^{N}$ is
**linear on the pair** $(x,y)$ (it is additive and homogeneous jointly):

$$
(\alpha x+\alpha' x')+(\alpha y+\alpha' y')
=\alpha(x+y)+\alpha'(x'+y').
\tag{1.2}
$$

**Proposition 1 (componentwise characterization).** For any index set
$I\subseteq[N]$, a vector $z$ satisfies $z_i=x_i+y_i$ for every $i\in I$ if and
only if $z$ restricted to $I$ equals $(x+y)$ restricted to $I$.

*Proof.* Both directions are immediate from (1.1): two vectors agree on $I$
exactly when every coordinate in $I$ agrees. $\square$

Proposition 1 is the license for **partitioned execution**: we may split $[N]$
into disjoint index sets, compute the sum independently on each, and the union
of the pieces is the unique correct output.

**Running example.** $N=5$, block size $B=4$. Then

$$
P=\left\lceil \frac{N}{B}\right\rceil=\left\lceil \frac54\right\rceil=2
$$

programs; program $1$ owns only the remainder index $4$.

### 0.1 Masked execution is a restriction map

A mask is an indicator vector

$$
m_i = \mathbf{1}\big[i\in I\big]
=
\begin{cases}
1, & i\in I,\\
0, & i\notin I.
\end{cases}
\tag{1.3}
$$

A masked load/store at address $i$ is, mathematically, the **restriction** of
the memory operation to $I$. Therefore a masked kernel that computes $z_i=x_i+y_i$
on each $i\in I$ and does nothing on $[N]\setminus I$ is exactly the map (1.1)
restricted to $I$. Combining over a partition $[N]=\bigsqcup_p I_p$ recovers the
full map. This is the only correctness argument needed for every masked kernel
in the repository.

### 0.2 Work, traffic, and the roofline

Each output coordinate performs

$$
\text{FLOPs}=1 \text{ add},
\qquad
\text{bytes}= \underbrace{b}_{\text{load }x_i}
+\underbrace{b}_{\text{load }y_i}
+\underbrace{b}_{\text{store }z_i}
=3b .
\tag{1.4}
$$

Thus, for the whole vector,

$$
F = N \text{ FLOPs},
\qquad
Q = 3Nb \text{ bytes}.
\tag{1.5}
$$

**Arithmetic intensity.** The ratio of work to traffic is

$$
\mathrm{AI}
=
\frac{F}{Q}
=
\frac{N}{3Nb}
=
\frac{1}{3b}
\ \text{FLOP/byte}.
\tag{1.6}
$$

For `float32`, $b=4$ and $\mathrm{AI}\approx 0.083$ FLOP/byte.

**Roofline bound.** Let $P_{\mathrm{peak}}$ be peak FLOP/s and $\mathrm{BW}$ the
achievable DRAM bandwidth (bytes/s). Any implementation satisfies

$$
t \;\ge\; \max\!\left(
\frac{F}{P_{\mathrm{peak}}},
\frac{Q}{\mathrm{BW}}
\right)
=
\max\!\left(
\frac{N}{P_{\mathrm{peak}}},
\frac{3Nb}{\mathrm{BW}}
\right).
\tag{1.7}
$$

A modern accelerator has a machine balance
$P_{\mathrm{peak}}/\mathrm{BW}\approx 10$–$100$ FLOP/byte. Since
$1/(3b)\ll P_{\mathrm{peak}}/\mathrm{BW}$, vector addition is
**memory-bound**: the first term in (1.7) is negligible and

$$
t \approx \frac{3Nb}{\mathrm{BW}}.
\tag{1.8}
$$

**Measured bandwidth.** If a kernel runs in $t$ seconds, its achieved bandwidth is

$$
\mathrm{GB/s}
=
\frac{3Nb}{t\cdot 10^{9}}.
\tag{1.9}
$$

This is exactly the `gbps` formula used in the benchmark cell. It is the number
to compare against the hardware spec sheet, not the raw runtime.

### 0.3 Warps, transactions, and coalescing

A CUDA warp contains $W=32$ threads (AMD wavefronts are 32 or 64). When the 32
lanes of a warp access 32 consecutive `float32` words, the hardware issues one
**128-byte transaction**. Hence

$$
\text{useful bytes per transaction}=32\cdot 4=128,
\qquad
\text{waste}=0
\tag{1.10}
$$

provided the base address is 128-byte aligned and the block length is a multiple
of 32. If lanes instead access strided addresses, each lane can trigger its own
transaction and effective bandwidth collapses by up to $32\times$. Triton
expresses this as a vector load `tl.load(ptr + offsets)`; the compiler coalesces
when `offsets` is a contiguous `tl.arange`.

**Corollary.** On a memory-bound kernel, the only performance levers are:
(a) move fewer bytes (fusion, lower precision), and
(b) move them in full coalesced transactions (alignment, contiguity).
Arithmetic micro-optimizations cannot beat the bandwidth wall (1.8).

### 0.4 Grid semantics: programs, blocks, warps

Triton exposes a **program** (CUDA thread block / CTA). A program has
`num_warps` warps, i.e. $32\,\texttt{num\_warps}$ threads. The grid

$$
\mathrm{grid}=(P,)
\tag{1.11}
$$

launches $P$ programs; inside a program `tl.program_id(0)` is a scalar in
$[P]$. A vector such as `tl.arange(0, B)` is distributed over the program's
threads by the compiler. Therefore the programmer reasons about **tiles of
work per program**, while the hardware maps lanes to threads and warps.

**Proposition 2 (blocked schedule covers $[N]$).** Let $B\ge 1$ and
$P=\lceil N/B\rceil$. Define

$$
I_p=\{pB+j: 0\le j<B\}\cap[N],
\qquad p\in[P].
\tag{1.12}
$$

Then $\{I_p\}_{p=0}^{P-1}$ is a partition of $[N]$.

*Proof.* For $i\in[N]$, the quotient–remainder theorem gives unique
$p=\lfloor i/B\rfloor$ and $j=i\bmod B$ with $0\le j<B$. Hence $i\in I_p$ and
$i\notin I_{p'}$ for $p'\ne p$. $\square$

### 0.5 Worked example ($N=5$, $B=4$)

| pid $p$ | offsets $pB+j$ | mask $j<N-pB$ | indices written |
|---|---|---|---|
| 0 | $\{0,1,2,3\}$ | $\{1,1,1,1\}$ | $\{0,1,2,3\}$ |
| 1 | $\{4,5,6,7\}$ | $\{1,0,0,0\}$ | $\{4\}$ |

The second tile has $B-(N\bmod B)=4-1=3$ inactive lanes. They must never be
loaded from nor stored to; `mask` enforces exactly that.

### Visual: theory board (method / logic)

![Whiteboard: naive 1-element programs vs blocked + mask tiling](docs/images/benchmark-vector-add.png)

*Theory board — naive grid size $N$ vs blocked tiles with `mask = offsets < N` (same algebra as the Triton kernel).*

### Hardware mental model

![Grid → Thread Block → Warp → Threads](docs/images/grid-block-warp-thread.jpeg)

![CUDA hierarchy: grid, blocks, warps of 32, SIMT](docs/images/cuda-execution-hierarchy.png)

### Theory board

![Whiteboard: naive 1-element programs vs blocked + mask tiling](docs/images/whiteboard-theory.jpg)

*Theory board — naive grid size $N$ vs blocked tiles with `mask = offsets < N`.*

# Part II — Proof

**Theorem (correctness of the blocked map).** Let $B\ge1$,
$P=\lceil N/B\rceil$, let $\mathrm{offsets}_p=pB+\{0,\dots,B-1\}$ and
$m_{p,j}=\mathbf{1}[pB+j<N]$. If a program executes $z_i\leftarrow x_i+y_i$
only at positions with $m_{p,j}=1$, then the union of all programs writes every
coordinate of $z=x+y$ exactly once and no out-of-range address.

*Proof.* By the quotient–remainder theorem every $i\in[N]$ has a unique
decomposition $i=pB+j$ with $0\le j<B$, hence belongs to exactly one tile $p$.
The predicate $m_{p,j}$ is true exactly when $0\le pB+j<N$, so valid lanes are
precisely the in-range indices. Uniqueness of the decomposition gives
disjointness, and the definition of $z_i$ gives the value. $\square$

**Corollary (method equivalence).** The naive map, the direct Triton kernel, and
the blocked/fused Triton kernel implement the same function
$z=x+y$.

*Proof.* Each is an instance of the theorem with a different tile size: $B=1$
for the naive map, $B=1$ for the direct kernel, and general $B$ for the fused
kernel. $\square$

# Part III — Step-by-step proof

1. **Choose the tile.** Fix $B$ (a power of two in Triton) and compute
   $P=\lceil N/B\rceil$. For $B=1$ this degenerates to one program per element.
2. **Identify the program.** Read $\mathrm{pid}\in[P]$ with
   `tl.program_id(0)`.
3. **Generate offsets.** Compute
   $\mathrm{offsets}=\mathrm{pid}\cdot B+\{0,\dots,B-1\}$.
4. **Build the mask.** Evaluate $m=\mathrm{offsets}<N$; invalid lanes are
   suppressed.
5. **Masked loads.** Load $x$ and $y$ at the valid offsets; `other` is
   irrelevant because the store is masked.
6. **Add.** Evaluate the single vector operation $x+y$ in registers.
7. **Masked store.** Write only valid lanes, so no address outside $[N]$ is
   touched.
8. **Cover.** Lemma/partition shows every index is handled once, so the
   concatenation of all program outputs is exactly $z$.
9. **Check the bound.** Count $3Nb$ bytes and $N$ FLOPs; conclude the kernel is
   memory-bound and the target is the bandwidth roofline.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



### Method 01 — Naive (Python map)

**Construction.** Identify each program id with a data index:

$$
\mathrm{pid}=i\in[N],
\qquad
z_i \leftarrow x_i+y_i.
\tag{1.13}
$$

**Proposition 3 (coverage and uniqueness).** The schedule (1.13) writes every
coordinate of $z$ exactly once, and the value written is the correct one.

*Proof.* The map $\mathrm{pid}\mapsto i$ is the identity on $[N]$, hence a
bijection. Every $i$ is therefore the image of exactly one program, and by
Definition 1 that program stores $x_i+y_i$. $\square$

**Worked check ($N=8$).** Eight programs; program $5$ loads $x_5,y_5$ and stores
$z_5=x_5+y_5$. No two programs touch the same address, so no synchronization is
required.

**Cost.** One program per element means

$$
\#\text{programs}=N,
\qquad
\text{work per program}=1\text{ add},
\qquad
\text{traffic per program}=3b\text{ bytes}.
\tag{1.14}
$$

The aggregate traffic is still $3Nb$ bytes, so the roofline bound (1.7) is
unchanged. The problem is **granularity**, not algebra.

### Why this schedule wastes the machine

1. **Warp under-utilization.** A program is scheduled as at least one warp
   ($W=32$ threads). If it owns a single scalar, at most one lane does useful
   work: compute utilization is $\le 1/W$.
2. **Transaction amplification.** A single scalar load cannot fill a 128-byte
   transaction; effective bandwidth is a fraction of peak.
3. **Launch/scheduling overhead.** The scheduler must dispatch $N$ blocks
   instead of $N/B$; for $N=10^{8}$ this is untenable.
4. **No instruction-level parallelism.** One add per program leaves the memory
   pipeline idle between requests.

These are performance defects, not correctness defects. The next section fixes
granularity by giving each program a contiguous tile.


In [ ]:
def naive_add_pseudo(x, y):
    """
    Elementwise add with one "pid" per output element.
    (See markdown cells above for the math.)
    """
    assert x.shape == y.shape
    n = x.numel()
    z = torch.empty_like(x)

    # Flatten so indexing mirrors the 1-D Triton grid
    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)
    z_flat = z.reshape(-1)

    for pid in range(n):
        z_flat[pid] = x_flat[pid] + y_flat[pid]

    return z


### Method 02 — Kernel-based (direct Triton)

The direct kernel is the literal SPMD translation of Method 01: one program owns
one output index, with no tile loop and no vector block. It exists to show the
mechanics of `program_id`, masks, and typed pointers before any blocking is
introduced.


In [ ]:
@triton.jit
def naive_add_kernel(x_ptr, y_ptr, output_ptr, n_elements):
    """Method 02: one program per element — the direct kernel translation."""
    pid = tl.program_id(axis=0)
    mask = pid < n_elements
    x = tl.load(x_ptr + pid, mask=mask)
    y = tl.load(y_ptr + pid, mask=mask)
    tl.store(output_ptr + pid, x + y, mask=mask)


def add_naive_kernel(x, y):
    output = torch.empty_like(x)
    n = output.numel()
    # Grid size N: one program for each element.
    naive_add_kernel[(n,)](x, y, output, n)
    return output


### Method 03 — Fused kernel-based (blocked + masked)

Fix a block size $B\in\mathbb{N}_{\ge 1}$ and let $P=\lceil N/B\rceil$.
Partition $[N]$ into contiguous tiles

$$
I_p
=
\{pB+j \mid 0\le j<B\}\cap[N],
\qquad p\in[P].
\tag{2.1}
$$

**Offsets (pre-mask).** Program $p$ generates the arithmetic progression

$$
\mathrm{offsets}_p
=
pB+\{0,1,\dots,B-1\}
=
\{pB,\,pB+1,\,\dots,\,pB+B-1\}.
\tag{2.2}
$$

**Mask (validity predicate).**

$$
m_{p,j}
=
\mathbf{1}\big[pB+j<N\big],
\qquad 0\le j<B.
\tag{2.3}
$$

**Proposition 4 (no out-of-bounds access).** If a load/store is executed only at
positions where $m_{p,j}=1$, then every accessed index lies in $[N]$.

*Proof.* $m_{p,j}=1\iff 0\le pB+j<N$; the lower bound holds because $p,j\ge0$. $\square$

**Proposition 5 (partition).** $\bigcup_{p\in[P]}I_p=[N]$ and
$I_p\cap I_{p'}=\varnothing$ for $p\ne p'$.

*Proof.* Same quotient–remainder argument as Proposition 2: each $i$ has unique
$p=\lfloor i/B\rfloor$ and $j=i\bmod B$. $\square$

**Corollary 6 (correctness of the blocked schedule).** If every program $p$
computes $z_i=x_i+y_i$ for all $i\in I_p$, then the assembled $z$ equals $x+y$.

*Proof.* Apply Proposition 5 and Proposition 1. $\square$

**Remainder size.** The last tile contains

$$
r
=
N-(P-1)B
=
\big((N-1)\bmod B\big)+1
\tag{2.4}
$$

valid elements and $B-r$ masked lanes. For $N=1000$, $B=256$:
$P=\lceil1000/256\rceil=4$, $r=1000-3\cdot256=232$, so the last tile has
$256-232=24$ inactive lanes.

**Teaching walk ($N=5$, $B=4$).**

| pid | offsets (pre-mask) | mask | writes |
|-----|--------------------|------|--------|
| 0 | $\{0,1,2,3\}$ | all true | $z[0:4]$ |
| 1 | $\{4,5,6,7\}$ | only index 4 | $z[4]$ |

### 2.1 Per-program work and balance

Each program performs at most $B$ adds and $3B$ memory accesses, so

$$
\text{work per program}\le B,
\qquad
\text{total work}=N,
\qquad
\text{grid}=P=\left\lceil\frac NB\right\rceil.
\tag{2.5}
$$

The schedule is balanced up to a remainder: for $N>0$,
$1\le |I_p|\le B$, with $|I_p|=B$ for every tile except the last and
$|I_{P-1}|=((N-1)\bmod B)+1$.
For $B\gg 1$ and $N\gg B$ the imbalance is at most one partial tile, i.e. a
relative imbalance $O(B/N)$.

### 2.2 Masked loads: what value do inactive lanes see?

Hardware predication suppresses the memory access for inactive lanes. The
`other` argument of `tl.load` supplies a value for those lanes so that the
subsequent arithmetic is well-defined. In vector addition the value is
irrelevant because the store is also masked; in reductions (softmax, layernorm)
the choice matters:

$$
\text{neutral element for } +:\quad \texttt{other}=0,
\qquad
\text{neutral element for } \max:\quad \texttt{other}=-\infty.
\tag{2.6}
$$

Choosing the wrong neutral element silently corrupts reductions — a recurring
bug in later notebooks.

### 2.3 Why power-of-two block sizes?

Triton lowers `tl.arange(0,B)` to a vector type whose length must be a
compile-time constant. Power-of-two $B$ maps cleanly onto warp/wavefront widths
and enables vectorized (128-bit) loads. For a power-of-two $B$ the remainder
tile is padded to a full vector, which is why masking is mandatory.


In [ ]:
def blocked_add_pseudo(x, y, BLOCK_SIZE: int = 1024):
    """
    Each pid owns a contiguous block of BLOCK_SIZE elements.
    (See markdown cells above for the math.)
    """
    assert x.shape == y.shape
    n = x.numel()
    z = torch.empty_like(x)

    x_flat = x.reshape(-1)
    y_flat = y.reshape(-1)
    z_flat = z.reshape(-1)

    num_pids = (n + BLOCK_SIZE - 1) // BLOCK_SIZE  # ceil division
    for pid in range(num_pids):
        block_start = pid * BLOCK_SIZE
        offsets = [block_start + i for i in range(BLOCK_SIZE)]

        for off in offsets:
            if off < n:  # mask for the last (partial) block
                z_flat[off] = x_flat[off] + y_flat[off]

    return z


#### Triton realization of the fused method

The blocked construction becomes SPMD code:

$$
\begin{aligned}
\mathrm{pid} &= \mathtt{tl.program\_id}(0),\\
\mathrm{offsets} &= \mathrm{pid}\cdot B + \mathtt{tl.arange}(0,B),\\
m &= \mathrm{offsets} < N,\\
z[\mathrm{offsets}] &\leftarrow x[\mathrm{offsets}] + y[\mathrm{offsets}]
\quad\text{(masked by $m$)}.
\end{aligned}
\tag{3.1}
$$

**Pointer arithmetic.** In Triton a pointer is a typed address. `x_ptr + offsets`
is element-wise pointer arithmetic:

$$
\&x_i = \&x_0 + i\cdot b,
\tag{3.2}
$$

so `tl.load(x_ptr + offsets, mask=m)` loads the vector
$(x_{\mathrm{offsets}_0},\dots,x_{\mathrm{offsets}_{B-1}})$ with invalid lanes
suppressed. `tl.store` is the dual operation.

**Equivalence theorem.** For every legal index $i$ there are unique $p,j$ with
$i=pB+j$ and $m_{p,j}=1$. Program $p$ therefore stores $x_i+y_i$ at $i$, and no
other program stores at $i$. Hence the Triton kernel computes exactly (1.1).

*Proof.* Combine Propositions 2, 4, 5 and Corollary 6. $\square$

```
|-- block 0 --|-- block 1 --|-- ... --| rem |
     pid=0         pid=1
```

| Kernel / stage | Program owns | Indexing |
|---|---|---|
| `naive_add_pseudo` | one $z_i$ | $\mathrm{pid}=i$ |
| `blocked_add_pseudo` | tile of $B$ | $pB+\{0..B-1\}$ + mask |
| `add_kernel` | same tile | `tl.program_id` / masked load-store |

### 3.1 From $P$ programs to the grid lambda

The launcher uses

$$
\mathrm{grid}
=
\left(\left\lceil \frac{N}{B}\right\rceil,1,1\right),
\tag{3.3}
$$

computed by `triton.cdiv(n_elements, BLOCK_SIZE)`. The `grid=lambda meta: ...`
form lets Triton read the autotuned `BLOCK_SIZE` before launch, so the grid and
the block size can never disagree.

### 3.2 Instruction-level view

Inside a program the compiler vectorizes the contiguous `tl.arange`:

- `float32` × 4 lanes = 128-bit `ld.global.v4.f32` per thread where possible;
- each warp then covers $32\times4=128$ elements per instruction;
- the `x + y` is a single vector add in registers.

Consequently the kernel approaches the bandwidth bound (1.8) as long as $N\gg B$
and the arrays are aligned. For small $N$ the launch latency $L$ dominates:

$$
t \approx L+\frac{3Nb}{\mathrm{BW}},
\qquad
\text{so}\quad
\mathrm{GB/s}\to 0 \text{ as } N\to 0.
\tag{3.4}
$$

This is why the benchmark sweeps $N=2^{12},\dots,2^{27}$: only the large sizes
measure the memory system rather than the launcher.

### 3.3 Vectorization and alignment

For peak efficiency the base pointers should be 16-byte aligned (PyTorch
allocations are) and `BLOCK_SIZE` should be a multiple of 4. If $B$ is a power of
two $\ge 128$, all of these hold. If the input is a non-contiguous view, Triton
receives strides instead of a flat pointer; this notebook assumes contiguous
1-D tensors, and the generic form would use `x_ptr + offsets * stride`.

**Bandwidth check.** After correctness (`allclose` vs `torch.add`), bench GB/s
with (1.9) — expect Triton ≈ PyTorch near peak DRAM.

### 3.4 Numerical note

Vector addition in IEEE-754 is exactly rounded: for each $i$,

$$
\mathrm{fl}(x_i+y_i)
=
(x_i+y_i)(1+\delta_i),
\qquad
|\delta_i|\le u,
\tag{3.5}
$$

where $u=2^{-24}\approx 5.96\times10^{-8}$ for `float32` and
$u=2^{-11}\approx 4.88\times10^{-4}$ for `float16`. There is no accumulation, so
the error does not grow with $N$. This is the simplest possible numerical
profile and a useful baseline against which reductions (softmax, layernorm) can
be compared.


In [ ]:
@triton.jit
def add_kernel(
    x_ptr,
    y_ptr,
    output_ptr,
    n_elements,
    BLOCK_SIZE: tl.constexpr,
):
    # Same blocked idea as blocked_add_pseudo:
    #   pid -> block_start -> offsets -> mask -> load / add / store
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    y = tl.load(y_ptr + offsets, mask=mask)
    tl.store(output_ptr + offsets, x + y, mask=mask)


def add(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    output = torch.empty_like(x)
    n_elements = output.numel()
    grid = lambda meta: (triton.cdiv(n_elements, meta["BLOCK_SIZE"]),)
    add_kernel[grid](x, y, output, n_elements, BLOCK_SIZE=1024)
    return output


### Pitfalls

- Forgetting the mask on the last tile → out-of-bounds loads.
- Using `BLOCK_SIZE` that is not a power of two can hurt Triton codegen; prefer 128/256/1024.
- Measuring tiny $N$ → launch overhead dominates; bench large $N$ for GB/s.
- Deriving bandwidth with $2Nb$ instead of $3Nb$ (forgetting the store).

### Self-check

1. Write offsets and mask for $N=5$, $B=4$ by hand; prove the partition lemma.
2. Starting from (1.5), derive the arithmetic intensity and show the kernel is memory-bound.
3. Change $B$ and re-run: does correctness still hold when $B\nmid N$? Why?


In [ ]:
def test_pseudo_matches_torch() -> None:
    """CPU-runnable: naive and blocked Python stages match torch.add."""
    torch.manual_seed(0)
    x = torch.rand(1000)
    y = torch.rand(1000)
    ref = x + y
    assert torch.allclose(naive_add_pseudo(x, y), ref)
    assert torch.allclose(blocked_add_pseudo(x, y, BLOCK_SIZE=128), ref)
    # Remainder case: n not divisible by BLOCK_SIZE
    assert torch.allclose(blocked_add_pseudo(x, y, BLOCK_SIZE=256), ref)


def test_correctness() -> None:
    require_gpu()
    torch.manual_seed(0)
    size = 98432
    x = torch.rand(size, device=DEVICE)
    y = torch.rand(size, device=DEVICE)
    out_torch = x + y
    out_triton = add(x, y)
    assert torch.allclose(out_torch, out_triton), torch.max(torch.abs(out_torch - out_triton)).item()
    print("correctness OK", torch.max(torch.abs(out_torch - out_triton)).item())


In [ ]:
@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=["size"],
        x_vals=[2**i for i in range(12, 28)],
        x_log=True,
        line_arg="provider",
        line_vals=["triton", "torch"],
        line_names=["Triton", "Torch"],
        styles=[("blue", "-"), ("green", "-")],
        ylabel="GB/s",
        plot_name="vector-add-performance",
        args={},
    )
)
def benchmark(size, provider):
    x = torch.rand(size, device=DEVICE, dtype=torch.float32)
    y = torch.rand(size, device=DEVICE, dtype=torch.float32)
    quantiles = [0.5, 0.2, 0.8]
    if provider == "torch":
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: x + y, quantiles=quantiles)
    if provider == "triton":
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: add(x, y), quantiles=quantiles)
    gbps = lambda ms: 3 * x.numel() * x.element_size() * 1e-9 / (ms * 1e-3)
    return gbps(ms), gbps(max_ms), gbps(min_ms)


In [ ]:
def test_three_methods() -> None:
    """Method 01 (Python) vs Method 02 (direct kernel) vs Method 03 (blocked)."""
    require_gpu()
    torch.manual_seed(0)
    x = torch.rand(4096, device=DEVICE)
    y = torch.rand(4096, device=DEVICE)
    ref = x + y
    assert torch.allclose(add_naive_kernel(x, y), ref)
    assert torch.allclose(add(x, y), ref)

    # Remainder case: N is not a multiple of the block size.
    x_small, y_small = x[:1000], y[:1000]
    assert torch.allclose(add_naive_kernel(x_small, y_small), x_small + y_small)
    assert torch.allclose(add(x_small, y_small), x_small + y_small)
    print("vector-add: methods 02 and 03 OK")
